# Case Study 2 — Support-Assistent über Dokumente (RAG)

**Fiktive Fallstudie VoltRide. Alle Daten sind erfunden.**

Läuft von oben nach unten durch. Alles liegt lokal im Colab-Dateisystem, keine
Drive-Anbindung.

| Zellen | Was passiert | Kosten |
|---|---|---|
| 1–5 | Setup, Daten, Chunking, Retrieval **und die komplette Retrieval-Diagnose** | 0 € |
| 6–8 | Preflight, Prompt-Varianten, Aufrufmechanik | 3 Testaufrufe |
| 9 | **Alle drei Läufe** in einem Rutsch, Bewertung, Export | ~0,01 € |
| 10–11 | Handcodierung, Vergleich, Diagramm, Kosten | 0 € |

**Zellen 10 und 11 lesen ausschließlich von der Festplatte.** Sie hängen an keiner
Variable aus einer früheren Zelle und funktionieren auch nach einem Neustart der
Laufzeit, solange die CSVs im Arbeitsordner liegen.

**Zwei getrennte Kennzahlen, nie eine gemeinsame:**

- **Trefferquote** auf den 17 beantwortbaren Fragen (14 `fakt` + 3 `mehrschritt`)
- **Ablehnungsquote** auf den 5 `luecke`-Fragen

Dazu, getrennt: **Beleg-Recall** — stand die belegende Textstelle überhaupt im Kontext?
Wenn nicht, ist eine falsche Antwort kein Modellfehler.

## 1 — Setup

Einmal ausführen. Installation dauert in Colab 1–2 Minuten.
Alle Ergebnisse landen im Ordner `cs2_arbeit` neben dem Notebook.

In [ ]:
# ============================================================================
# ZELLE 1 — Installation, Konfiguration, Arbeitsordner (lokal)
# ============================================================================
import subprocess, sys

def _pip(*p):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=False)

try:
    import sentence_transformers  # noqa
except ImportError:
    _pip("sentence-transformers")
try:
    import openai  # noqa
except ImportError:
    _pip("openai")

import os, re, json, time, hashlib, textwrap, random
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

# ---------------------------------------------------------------------------
# KONFIGURATION — hier und nur hier wird gedreht
# ---------------------------------------------------------------------------
CFG = dict(
    MODELL       = "openai/gpt-oss-20b",
    BASE_URL     = "https://api.groq.com/openai/v1",
    MAX_TOKENS   = 1600,      # Reasoning-Modell: ~70 % der Token gehen ins Nachdenken
    TEMPERATURE  = 0.0,
    REASONING    = "low",     # wird im Preflight verifiziert
    TOP_K        = 3,         # wie viele Abschnitte ins Prompt
    MAX_PRO_DOK  = 3,         # max. Abschnitte aus DEMSELBEN Dokument (3 = keine Grenze)
    EMB_MODELL   = "sentence-transformers/all-MiniLM-L6-v2",
    EMB_ALT      = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    TABELLEN_ZEILENWEISE = False,   # True: jede Tabellenzeile wird ein eigener Abschnitt
    MAX_VERSUCHE = 3,
    SEED         = 42,
)

random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])

# Arbeitsordner: lokal, neben dem Notebook. Kein Drive.
ARBEIT = Path("cs2_arbeit")
ARBEIT.mkdir(exist_ok=True)
CACHE_PFAD = ARBEIT / "rag_cache.json"

print("Arbeitsordner:", ARBEIT.resolve())
print("Python", sys.version.split()[0])
print("\nHinweis: Colab loescht diesen Ordner beim Neustart der Laufzeit.")
print("Nach jedem Lauf die CSVs aus cs2_arbeit herunterladen, dann geht nichts verloren.")

## 2 — Daten laden

**Ordnerstruktur ist egal.** Die Zelle sucht die sieben Dateien überall unter `/content`
und im Arbeitsverzeichnis — flach hochgeladen, in `docs/`, oder gemischt.

Fehlt etwas, öffnet sich ein Datei-Dialog. Dort die **sieben Einzeldateien** auswählen
(Mehrfachauswahl mit ⌘-Klick), keinen Ordner:

```
01_preise_und_tarife.md   02_support_playbook.md   03_release_notes_app.md
04_geschaeftsgebiet.md    05_faq_nutzer.md         06_produktprinzipien.md
eval_fragen_rag.csv
```

In [ ]:
# ============================================================================
# ZELLE 2 — Daten finden und laden
# ============================================================================
SUCH_WURZELN = [Path("/content"), Path.cwd()]
UEBERSPRINGEN = {"drive", "sample_data", ".git", "__pycache__",
                 ".ipynb_checkpoints", ".config", "node_modules"}
STICHWORT = {"preise": "01", "tarif": "01", "playbook": "02", "support": "02",
             "release": "03", "app": "03", "geschaeft": "04", "geschäft": "04",
             "gebiet": "04", "faq": "05", "nutzer": "05", "prinzip": "06"}

def dok_id_aus_name(name):
    m = re.match(r"0?([1-6])[_\-. ]", name)
    if m:
        return f"0{m.group(1)}"
    klein = name.lower()
    for wort, num in STICHWORT.items():
        if wort in klein:
            return num
    return None

def sammle_dateien(wurzeln=SUCH_WURZELN, max_tiefe=4):
    dok, csv_pfad, gesehen = {}, None, set()
    for w in wurzeln:
        try:
            if not w.is_dir():
                continue
            basis = w.resolve()
            if basis in gesehen:
                continue
            gesehen.add(basis)
            for ordner, unter, dateien in os.walk(basis):
                if len(Path(ordner).relative_to(basis).parts) >= max_tiefe:
                    unter[:] = []
                unter[:] = [u for u in unter if u not in UEBERSPRINGEN]
                for name in dateien:
                    if name == "eval_fragen_rag.csv" and csv_pfad is None:
                        csv_pfad = Path(ordner) / name
                    elif name.lower().endswith(".md"):
                        d = dok_id_aus_name(name)
                        if d and d not in dok:
                            dok[d] = Path(ordner) / name
        except (OSError, PermissionError):
            continue
    return dok, csv_pfad

def in_colab():
    try:
        import google.colab  # noqa
        return True
    except ImportError:
        return False

def was_fehlt(dok, csv_pfad):
    f = [f"0{i}_*.md" for i in range(1, 7) if f"0{i}" not in dok]
    if csv_pfad is None:
        f.append("eval_fragen_rag.csv")
    return f

dok_pfade, csv_pfad = sammle_dateien()
for runde in range(3):
    fehlt = was_fehlt(dok_pfade, csv_pfad)
    if not fehlt:
        break
    print(f"Noch fehlend ({len(fehlt)}): {', '.join(fehlt)}")
    if not in_colab():
        print("Dateien in den Arbeitsordner legen und Zelle erneut ausfuehren.")
        break
    print("Einzeldateien im Dialog auswaehlen — keinen Ordner.\n")
    from google.colab import files
    ziel = Path("/content/voltride_daten"); ziel.mkdir(parents=True, exist_ok=True)
    for name, inhalt in files.upload().items():
        (ziel / Path(name).name).write_bytes(inhalt)
        print("  gespeichert:", ziel / Path(name).name)
    dok_pfade, csv_pfad = sammle_dateien()

fehlt = was_fehlt(dok_pfade, csv_pfad)
if fehlt:
    raise SystemExit("Es fehlen weiterhin: " + ", ".join(fehlt))

DOKUMENTE = {d: p.read_text(encoding="utf-8") for d, p in sorted(dok_pfade.items())}
eval_df = pd.read_csv(csv_pfad, dtype=str).fillna("")
eval_df.columns = [c.strip() for c in eval_df.columns]

print("Gefunden:")
for d, p in sorted(dok_pfade.items()):
    print(f"  {d} <- {p}")
print(f"  csv <- {csv_pfad}")

print("\nVerteilung der Fragetypen (Verteilung VOR Kennzahl):")
print(eval_df["typ"].value_counts().to_string())

annahmen = {
    "6 Dokumente":       len(DOKUMENTE) == 6,
    "25 Fragen":         len(eval_df) == 25,
    "IDs eindeutig":     eval_df["id"].is_unique,
    "keine leere Frage": (eval_df["frage"].str.len() > 0).all(),
    "Typen wie erwartet": set(eval_df["typ"]) == {"fakt", "luecke", "mehrschritt",
                                                  "falle", "widerspruch"},
}
for name, ok in annahmen.items():
    print(f"  {'OK  ' if ok else 'FEHL'} {name}")
assert all(annahmen.values()), "Sanity-Check fehlgeschlagen — nicht weiterrechnen."

BEANTWORTBAR = eval_df.loc[eval_df["typ"].isin(["fakt", "mehrschritt"]), "id"].tolist()
LUECKEN      = eval_df.loc[eval_df["typ"] == "luecke", "id"].tolist()
WIDERSPRUCH  = eval_df.loc[eval_df["typ"] == "widerspruch", "id"].tolist()
FALLE        = eval_df.loc[eval_df["typ"] == "falle", "id"].tolist()
print(f"\nbeantwortbar n={len(BEANTWORTBAR)} | luecke n={len(LUECKEN)} | "
      f"widerspruch n={len(WIDERSPRUCH)} | falle n={len(FALLE)}")

## 3 — Dokumente in Abschnitte zerlegen

Geschnitten wird an den `##`-Überschriften. Der Text **vor** der ersten Überschrift wird
mitgenommen — in `01_preise_und_tarife.md` steht dort „Gültig für alle Städte außer
Aarburg", die Zeile, die Fangfrage Q21 entscheidet.

`CFG["TABELLEN_ZEILENWEISE"]` schneidet Tabellen zusätzlich zeilenweise. Das ist ein
eigener Eingriff und gehört als eigener Lauf protokolliert.

In [ ]:
# ============================================================================
# ZELLE 3 — Chunking
# ============================================================================
MAX_CHUNK_ZEICHEN = 1200

def tabelle_aufteilen(text, min_zeilen=3):
    """Zerlegt Markdown-Tabellen in Einzelzeilen, Kopfzeile bleibt bei jeder."""
    bloecke, puffer, tab = [], [], []

    def leeren():
        if not tab:
            return
        kopf = tab[:2] if len(tab) >= 2 and set(tab[1].replace("|", "").strip()) <= set("-: ") else []
        daten = tab[len(kopf):]
        if len(daten) >= min_zeilen:
            bloecke.extend("\n".join(kopf + [z]) for z in daten)
        else:
            bloecke.append("\n".join(tab))
        tab.clear()

    for z in text.split("\n"):
        if z.lstrip().startswith("|"):
            if puffer:
                bloecke.append("\n".join(puffer).strip()); puffer = []
            tab.append(z)
        else:
            leeren(); puffer.append(z)
    leeren()
    if any(x.strip() for x in puffer):
        bloecke.append("\n".join(puffer).strip())
    return [b for b in bloecke if b.strip()]

def zerlege_dokument(dok_id, text, max_zeichen=MAX_CHUNK_ZEICHEN):
    zeilen = text.split("\n")
    titel_dok = next((z.lstrip("# ").strip() for z in zeilen if z.startswith("# ")),
                     f"Dokument {dok_id}")
    abschnitte, aktuell, titel = [], [], "Kopf"
    for z in zeilen:
        if z.startswith("## "):
            if any(x.strip() for x in aktuell):
                abschnitte.append((titel, "\n".join(aktuell).strip()))
            titel, aktuell = z[3:].strip(), []
        else:
            aktuell.append(z)
    if any(x.strip() for x in aktuell):
        abschnitte.append((titel, "\n".join(aktuell).strip()))

    chunks = []
    for titel, inhalt in abschnitte:
        if CFG["TABELLEN_ZEILENWEISE"] and "|" in inhalt:
            teile = tabelle_aufteilen(inhalt)
        elif len(inhalt) <= max_zeichen:
            teile = [inhalt]
        else:
            teile, puffer = [], []
            for block in inhalt.split("\n\n"):
                if puffer and sum(len(b) for b in puffer) + len(block) > max_zeichen \
                        and not block.lstrip().startswith("|"):
                    teile.append("\n\n".join(puffer)); puffer = [block]
                else:
                    puffer.append(block)
            if puffer:
                teile.append("\n\n".join(puffer))
        for i, t in enumerate(teile):
            chunks.append(dict(chunk_id=f"{dok_id}#{len(chunks):02d}", dok=dok_id,
                               dok_titel=titel_dok,
                               abschnitt=titel + (f" ({i+1})" if len(teile) > 1 else ""),
                               text=t))
    return chunks

CHUNKS = []
for d in sorted(DOKUMENTE):
    CHUNKS.extend(zerlege_dokument(d, DOKUMENTE[d]))

chunk_df = pd.DataFrame(CHUNKS)
chunk_df["zeichen"] = chunk_df["text"].str.len()
print(f"{len(chunk_df)} Abschnitte, Tabellen zeilenweise = {CFG['TABELLEN_ZEILENWEISE']}\n")
print(chunk_df.groupby("dok").agg(abschnitte=("chunk_id", "count"),
                                  zeichen=("zeichen", "sum"),
                                  laengster=("zeichen", "max")).to_string())
assert (chunk_df["zeichen"] > 0).all() and chunk_df["chunk_id"].is_unique

## 4 — Embeddings und Retrieval

Läuft lokal auf der CPU, kostet nichts, braucht keinen API-Key.

`CFG["MAX_PRO_DOK"]` begrenzt, wie viele Abschnitte aus demselben Dokument in den Kontext
dürfen. Ohne Grenze liefert reines Kosinus regelmäßig dreimal dasselbe Dokument — der
Kontext ist dann faktisch top-1.

In [ ]:
# ============================================================================
# ZELLE 4 — Embeddings, Index, Suche
# ============================================================================
from sentence_transformers import SentenceTransformer

_MODELLE = {}

def lade_emb(name):
    if name not in _MODELLE:
        print(f"lade {name} ...")
        _MODELLE[name] = SentenceTransformer(name)
    return _MODELLE[name]

def baue_index(chunks, modell_name):
    m = lade_emb(modell_name)
    texte = [f"{c['dok_titel']} — {c['abschnitt']}\n{c['text']}" for c in chunks]
    V = np.asarray(m.encode(texte, normalize_embeddings=True, show_progress_bar=False),
                   dtype=np.float32)
    return dict(modell=modell_name, vektoren=V, chunks=chunks)

def suche(frage, index=None, top_k=None, max_pro_dok=None):
    index = index if index is not None else INDEX
    top_k = top_k or CFG["TOP_K"]
    max_pro_dok = max_pro_dok or CFG["MAX_PRO_DOK"]
    m = lade_emb(index["modell"])
    q = np.asarray(m.encode([frage], normalize_embeddings=True), dtype=np.float32)[0]
    scores = index["vektoren"] @ q
    treffer, pro_dok = [], {}
    for i in np.argsort(-scores):
        c = index["chunks"][i]
        if pro_dok.get(c["dok"], 0) >= max_pro_dok:
            continue
        pro_dok[c["dok"]] = pro_dok.get(c["dok"], 0) + 1
        treffer.append(dict(c, score=float(scores[i])))
        if len(treffer) == top_k:
            break
    return treffer

INDEX = baue_index(CHUNKS, CFG["EMB_MODELL"])
print(f"Index: {INDEX['vektoren'].shape[0]} Abschnitte x {INDEX['vektoren'].shape[1]} Dim")
print("\nProbe — 'Wie viele Raeder hat VoltRide in Talheide?'")
for t in suche("Wie viele Räder hat VoltRide in Talheide?"):
    print(f"  {t['score']:.3f}  {t['dok']}·{t['abschnitt']}")

## 5 — Retrieval-Diagnose, **ohne einen einzigen API-Aufruf**

Diese Zelle ist der wichtigste kostenlose Schritt. Sie definiert die Prüfmuster und
beantwortet drei Fragen:

1. War das erwartete **Dokument** unter den Top-3? *(weicher Maßstab)*
2. Stand der **Beleg** selbst im Kontext? *(harter Maßstab — Dokument 04 hat mehrere
   Abschnitte, und nur einer enthält die Städtetabelle)*
3. Was hat sich gegenüber dem letzten Durchlauf **verschoben**? Zwei Konfigurationen
   können denselben Recall haben und an völlig anderen Fragen scheitern. Genau so
   versteckt sich eine Regression hinter einer Prozentzahl.

Hier drehst du an `TABELLEN_ZEILENWEISE`, `MAX_PRO_DOK`, `TOP_K` oder am Embedding-Modell
und siehst sofort, ob es hilft — **bevor** du 25 Aufrufe bezahlst.

In [ ]:
# ============================================================================
# ZELLE 5 — Pruefmuster und Retrieval-Diagnose (kostenlos)
# ============================================================================
def norm(t):
    t = str(t).lower().replace(" ", " ").replace(" ", " ")
    t = t.replace("€", " eur ")
    t = re.sub(r"(?<=\d),(?=\d)", ".", t)     # 0,12 -> 0.12
    return re.sub(r"\s+", " ", t)

NEIN = r"(nein|nicht|kein)"

# muss  : fehlt eines -> Antwort ist FALSCH
# bonus : fehlt eines -> Antwort ist richtig, aber UNVOLLSTAENDIG
# beleg : woran der Beleg im KONTEXT erkannt wird (Standard: muss).
#         Noetig bei Rechenaufgaben — "5,40" steht in keinem Dokument.
# tabu  : darf nicht vorkommen
PRUEF = {
    "Q01": dict(muss=[r"\b1(\.0{1,2})?\s*eur"]),
    "Q02": dict(muss=[r"\b0\.12\s*eur"]),
    "Q03": dict(muss=[r"\b45(\.0{1,2})?\s*eur"], bonus=[r"\b24\s*(stunden|std|h)\b"]),
    "Q04": dict(muss=[NEIN], tabu=[r"^\s*ja[,. ]"]),
    "Q05": dict(muss=[NEIN], tabu=[r"^\s*ja[,. ]"],
                bonus=[r"(kein (termin|datum)|spaeteres release|späteres release|noch nicht|geplant|vorgesehen)"]),
    "Q06": dict(muss=[r"\b25\s*km"]),
    "Q07": dict(muss=[r"(\b1\s*(stunde|std|h)\b|einer stunde|60 minuten)"]),
    "Q08": dict(muss=[r"(teamlead|team lead|team-lead|freigabe|genehmigung)"],
                tabu=[r"^\s*ja[,. ]"], bonus=[NEIN]),
    "Q09": dict(muss=[r"aarburg", r"\b0\.19"]),
    "Q10": dict(muss=[NEIN, r"(nacht|00\s*[-–:.]?\s*0?0?\s*(bis|-|–)\s*0?5|zwischen 0?0 und 0?5)"],
                tabu=[r"^\s*ja[,. ]"]),
    "Q11": dict(muss=[r"(fotoupload|foto-upload|foto upload)", r"android\s*15"]),
    "Q12": dict(muss=[r"\b620\b"]),
    "Q13": dict(muss=[r"\b20(\.0{1,2})?\s*eur"]),
    "Q14": dict(muss=[NEIN], tabu=[r"^\s*ja[,. ]"]),
    "Q15": dict(muss=[NEIN], tabu=[r"^\s*ja[,. ]"]),
    "Q22": dict(muss=[r"\b5\.4\d?\s*eur"],
                beleg=[r"\b1(\.0{1,2})?\s*eur", r"\b0\.22\s*eur"]),
    "Q25": dict(muss=[NEIN, r"(freigegeben|freigabe|genehmigt)"], tabu=[r"^\s*ja[,. ]"]),
    "Q23": dict(muss=[r"osterbr", r"(prinzip|playbook|gesperrt|sperr)"], braucht_widerspruch=True),
    "Q24": dict(muss=[r"sicherheit", r"(verfuegbarkeit|verfügbarkeit|400\s*meter|400\s*m\b)"],
                braucht_widerspruch=True),
    "Q21": dict(nur_manuell=True),
}
fehlend = set(BEANTWORTBAR + WIDERSPRUCH) - set(PRUEF)
assert not fehlend, f"Pruefmuster fehlen fuer: {sorted(fehlend)}"

def erwartete_doks(quelle_dok):
    s = str(quelle_dok).strip()
    return set() if s in ("", "-", "nan") else {t.strip().zfill(2) for t in s.split("+") if t.strip()}

def beleg_im_kontext(qid, kontext):
    """True/False, ob der Beleg im gelieferten Kontext stand. None = nicht pruefbar."""
    regel = PRUEF.get(qid, {})
    muster = regel.get("beleg") or regel.get("muss", [])
    if not muster or regel.get("nur_manuell"):
        return None
    k = norm(kontext)
    return all(re.search(p, k) for p in muster)

def retrieval_diagnose(index=None, df=None):
    df = eval_df if df is None else df
    zeilen = []
    for _, r in df.iterrows():
        t = suche(r["frage"], index=index)
        soll, ist = erwartete_doks(r["quelle_dok"]), [x["dok"] for x in t]
        zeilen.append(dict(
            id=r["id"], typ=r["typ"], soll="+".join(sorted(soll)) or "-",
            abschnitte=" | ".join(f'{x["dok"]}·{x["abschnitt"]}' for x in t),
            n_doks=len(set(ist)),
            dok_any=bool(soll & set(ist)) if soll else None,
            beleg=beleg_im_kontext(r["id"], "\n".join(x["text"] for x in t)),
        ))
    return pd.DataFrame(zeilen)

diag = retrieval_diagnose()
# Spalten koennen None enthalten -> object dtype. Fuer Filter und Mittelwert
# ausdruecklich nach bool wandeln, sonst wird "~" zu einer Ganzzahl-Operation.
dok_m = diag[diag["dok_any"].notna()].copy()
dok_m["dok_any"] = dok_m["dok_any"].astype(bool)
bel_m = diag[diag["beleg"].notna()].copy()
bel_m["beleg"] = bel_m["beleg"].astype(bool)

print(f"Konfiguration: top_k={CFG['TOP_K']} · max_pro_dok={CFG['MAX_PRO_DOK']} · "
      f"tabellen_zeilenweise={CFG['TABELLEN_ZEILENWEISE']}")
print(f"Embedding: {CFG['EMB_MODELL'].split('/')[-1]}\n")
print(f"  Dokument-Recall (weich, n={len(dok_m)}) : {dok_m['dok_any'].mean():.0%}")
print(f"  Beleg-Recall    (hart,  n={len(bel_m)}) : {bel_m['beleg'].mean():.0%}   <- der ehrliche Wert")
print(f"  Fragen mit nur EINEM Dokument in den Top-3: "
      f"{(diag['n_doks'] == 1).sum()} von {len(diag)}")

print("\nOhne Beleg im Kontext — hier kann das Modell gar nicht richtig antworten:")
for _, r in bel_m[~bel_m["beleg"]].iterrows():
    print(f"  {r['id']} [{r['typ']}]  bekommen: {r['abschnitte']}")

# --- Zusammensetzung vor Prozentzahl: was hat sich seit dem letzten Mal verschoben?
VERGLEICH = ARBEIT / "retrieval_letzter.csv"
if VERGLEICH.exists():
    alt = pd.read_csv(VERGLEICH, dtype=str).set_index("id")["beleg"].map(
        lambda x: str(x).strip().lower() == "true").to_dict()
    besser     = [r["id"] for _, r in bel_m.iterrows() if r["beleg"] and alt.get(r["id"]) is False]
    schlechter = [r["id"] for _, r in bel_m.iterrows() if not r["beleg"] and alt.get(r["id"]) is True]
    alt_quote = sum(1 for i in bel_m["id"] if alt.get(i)) / len(bel_m)
    print(f"\nGegenueber dem letzten Durchlauf: {alt_quote:.0%} -> {bel_m['beleg'].mean():.0%}")
    print(f"  repariert ({len(besser)}): {', '.join(besser) or '—'}")
    print(f"  kaputt    ({len(schlechter)}): {', '.join(schlechter) or '—'}")
    if schlechter:
        print("  >>> REGRESSION. Die Prozentzahl allein haette das verdeckt.")
else:
    print(f"\n(kein Vergleichsstand — {VERGLEICH.name} wird jetzt angelegt)")
diag.to_csv(VERGLEICH, index=False)
diag.to_csv(ARBEIT / "retrieval_diagnose.csv", index=False, sep=";", encoding="utf-8-sig")
print(f"-> {ARBEIT / 'retrieval_diagnose.csv'}")

## 6 — API-Client und Preflight

Ab hier kostet es. Der Preflight prüft **vor** dem ersten vollen Lauf, welche Parameter
das Modell wirklich akzeptiert, statt sich auf Dokumentation zu verlassen.

Groq-Doku, Stand August 2026: `openai/gpt-oss-20b` kann `json_schema` mit `strict: true`
und `false` sowie `reasoning_effort` in `low|medium|high`. `reasoning_format` kann es
**nicht** — das Gegenstück heißt `include_reasoning`. Der Preflight misst es trotzdem nach.

Aus Case Study 1: **`strict: true` ist keine Garantie.** Die eigene Validierung in Zelle 8
bleibt der Maßstab, nicht das Schema.

In [ ]:
# ============================================================================
# ZELLE 6 — Client und Preflight
# ============================================================================
from openai import OpenAI

def hole_key():
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
        if k:
            return k.strip()
    except Exception:
        pass
    k = os.environ.get("GROQ_API_KEY")
    if k:
        return k.strip()
    import getpass
    return getpass.getpass("GROQ_API_KEY: ").strip()

client = OpenAI(api_key=hole_key(), base_url=CFG["BASE_URL"])

ANTWORT_SCHEMA = {
    "name": "support_antwort",
    "schema": {
        "type": "object",
        "properties": {
            "status":      {"type": "string",
                            "enum": ["beantwortet", "unsicher", "nicht_im_dokument"]},
            "antwort":     {"type": "string"},
            "quellen":     {"type": "array", "items": {"type": "string"}},
            "widerspruch": {"type": "boolean"},
        },
        "required": ["status", "antwort", "quellen", "widerspruch"],
        "additionalProperties": False,
    },
}

def roher_aufruf(messages, extra=None, response_format=None, max_tokens=None):
    kw = dict(model=CFG["MODELL"], messages=messages,
              max_tokens=max_tokens or CFG["MAX_TOKENS"], temperature=CFG["TEMPERATURE"])
    if response_format:
        kw["response_format"] = response_format
    if extra:
        kw["extra_body"] = extra
    return client.chat.completions.create(**kw)

probe = [{"role": "system", "content": "Antworte ausschliesslich mit JSON."},
         {"role": "user", "content": '{"status":"beantwortet","antwort":"ok",'
                                     '"quellen":["01"],"widerspruch":false} zurueckgeben'}]
tests = [
    ("blank",              dict()),
    ("reasoning_effort",   dict(extra={"reasoning_effort": CFG["REASONING"]})),
    ("json_object",        dict(response_format={"type": "json_object"})),
    ("json_schema_lose",   dict(response_format={"type": "json_schema",
                                "json_schema": {**ANTWORT_SCHEMA, "strict": False}})),
    ("json_schema_strict", dict(response_format={"type": "json_schema",
                                "json_schema": {**ANTWORT_SCHEMA, "strict": True}})),
]
zeilen = []
for name, kw in tests:
    try:
        r = roher_aufruf(probe, **kw)
        txt = (r.choices[0].message.content or "").strip()
        zeilen.append(dict(test=name, ok=bool(txt), out=r.usage.completion_tokens,
                           probe=txt[:60].replace("\n", " "), fehler=""))
    except Exception as e:
        zeilen.append(dict(test=name, ok=False, out=None, probe="",
                           fehler=f"{type(e).__name__}: {str(e)[:80]}"))
    time.sleep(1.0)
pre = pd.DataFrame(zeilen)
print(pre.to_string(index=False))

def _ok(n):
    z = pre[pre.test == n]
    return bool(len(z) and z.iloc[0]["ok"])

if not any(pre["ok"]):
    raise SystemExit("Kein Aufruf hat funktioniert. Key oder Kontingent pruefen.")

if _ok("json_schema_strict"):
    RESPONSE_FORMAT = {"type": "json_schema", "json_schema": {**ANTWORT_SCHEMA, "strict": True}}
elif _ok("json_schema_lose"):
    RESPONSE_FORMAT = {"type": "json_schema", "json_schema": {**ANTWORT_SCHEMA, "strict": False}}
elif _ok("json_object"):
    RESPONSE_FORMAT = {"type": "json_object"}
else:
    RESPONSE_FORMAT = None
EXTRA = {"reasoning_effort": CFG["REASONING"]} if _ok("reasoning_effort") else {}
print("\ngewaehlt:", (RESPONSE_FORMAT or {}).get("type", "keins"), "| extra:", EXTRA)

## 7 — Die drei Prompt-Varianten

Der Zielkonflikt steckt in genau einem Absatz, der sich unterscheidet. Alles andere bleibt
identisch — sonst ist der Vergleich wertlos.

| Variante | Eingriff | Erwartung, **vor** dem Lauf notiert |
|---|---|---|
| `basis` | Ablehnen erlaubt, nicht betont | hohe Trefferquote, niedrige Ablehnungsquote |
| `streng` | nur antworten, wenn belegt oder einfach errechenbar | Ablehnung steigt, Treffer sinkt leicht |
| `sehr_streng` | jede Zahl muss wörtlich im Ausschnitt stehen | `mehrschritt` bricht ein |

In [ ]:
# ============================================================================
# ZELLE 7 — Prompt-Varianten
# ============================================================================
BASIS_ROLLE = """Du bist ein Assistent fuer den Support Level 1 von VoltRide, einem E-Bike-Sharing-Dienst.
Du beantwortest Fragen ausschliesslich auf Basis der vorgelegten Dokumentausschnitte.
Du verwendest kein Weltwissen und keine Annahmen ueber VoltRide.

Antworte als JSON-Objekt mit genau diesen Feldern:
  "status"      : "beantwortet" | "unsicher" | "nicht_im_dokument"
  "antwort"     : ein bis drei Saetze, deutsch, mit konkreten Zahlen wenn vorhanden
  "quellen"     : Liste der Dokumentnummern, z.B. ["01","05"]
  "widerspruch" : true, wenn sich die Ausschnitte in dieser Frage widersprechen, sonst false

Regeln fuer "status":
  "beantwortet"       - die Ausschnitte belegen die Antwort vollstaendig
  "unsicher"          - die Ausschnitte belegen nur einen Teil der Frage
  "nicht_im_dokument" - die Ausschnitte belegen die Antwort nicht

Wenn "widerspruch" true ist, nenne in "antwort" BEIDE Seiten und ihre Quellen."""

ABLEHNUNGSREGEL = {
    "basis": """Wenn die Ausschnitte die Frage nicht abdecken, setze "status" auf "nicht_im_dokument"
und schreibe in "antwort", dass das in den Dokumenten nicht steht.""",

    "streng": """Antworte nur, wenn die Antwort woertlich in einem Ausschnitt steht oder sich durch eine
einfache Rechnung direkt daraus ergibt. Rate nicht, schaetze nicht, kombiniere nichts
Ungesichertes. Im Zweifel "nicht_im_dokument". Eine falsche Zahl ist schlimmer als keine
Zahl - der Support gibt diese Auskunft an zahlende Kunden weiter.""",

    "sehr_streng": """Jede Zahl und jede Frist in deiner Antwort muss woertlich in einem der Ausschnitte stehen.
Ist auch nur ein Teil der Frage nicht belegt, lautet "status" "nicht_im_dokument".
Leite nichts ab, rechne nichts aus, schliesse nicht von einer Stadt auf eine andere und
nicht von einem Tarif auf einen anderen. Im Zweifel immer "nicht_im_dokument".""",
}

def baue_prompt(frage, treffer, variante):
    ausschnitte = "\n\n".join(
        f'[Dok {t["dok"]} · {t["dok_titel"]} · Abschnitt "{t["abschnitt"]}"]\n{t["text"]}'
        for t in treffer)
    return [{"role": "system", "content": BASIS_ROLLE + "\n\n" + ABLEHNUNGSREGEL[variante]},
            {"role": "user", "content": f"Dokumentausschnitte:\n\n{ausschnitte}\n\n"
                                        f"----\nFrage des Support-Mitarbeiters: {frage}\n\n"
                                        f"Antworte als JSON."}]

for v, t in ABLEHNUNGSREGEL.items():
    print(f"--- {v} ---\n{t}\n")

## 8 — Aufruf, eigene Validierung, Bewertung

Nur Definitionen, hier passiert noch nichts. Vier Regeln aus Case Study 1:

- **Eigene Validierung nach dem Schema.** Jede Schemaverletzung wird protokolliert — auch
  dann, wenn ein Wiederholungsversuch sie geheilt hat.
- **Nur Erfolge cachen.** Ein Fehlschlag im Cache erzeugt still falsche Messwerte.
- **Fehlgeschlagene Aufrufe sind keine Vorhersage.** `ok=False`, leere Felder, und
  Kennzahlen laufen nur über erfolgreiche Zeilen — mit sichtbarer Erfolgsquote.
- **Drei Versuche, längere Wartezeit bei Rate-Limits.**

In [ ]:
# ============================================================================
# ZELLE 8 — Aufruf, Validierung, Cache, Bewertung, Kennzahlen (nur Definitionen)
# ============================================================================
CACHE = json.loads(CACHE_PFAD.read_text()) if CACHE_PFAD.exists() else {}
print(f"Cache: {len(CACHE)} erfolgreiche Eintraege")

ERLAUBTE_STATUS = {"beantwortet", "unsicher", "nicht_im_dokument"}
ABGELEHNT, VORBEHALT = {"nicht_im_dokument"}, {"unsicher"}

def cache_key(variante, qid, messages):
    roh = json.dumps([CFG["MODELL"], CFG["EMB_MODELL"], CFG["TOP_K"], CFG["MAX_PRO_DOK"],
                      CFG["TABELLEN_ZEILENWEISE"], variante, qid, messages],
                     sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(roh.encode()).hexdigest()[:32]

def json_aus_text(txt):
    if not txt:
        return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", txt.strip(), flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    tiefe, start = 0, None
    for i, c in enumerate(t):
        if c == "{":
            if tiefe == 0:
                start = i
            tiefe += 1
        elif c == "}":
            tiefe -= 1
            if tiefe == 0 and start is not None:
                try:
                    return json.loads(t[start:i + 1])
                except Exception:
                    start = None
    return None

def validiere(obj):
    v = []
    if not isinstance(obj, dict):
        return None, ["kein_objekt"]
    status = obj.get("status")
    if not isinstance(status, str) or status.strip().lower() not in ERLAUBTE_STATUS:
        v.append(f"status_ungueltig:{status!r}"); status = None
    else:
        status = status.strip().lower()
    antwort = obj.get("antwort")
    if not isinstance(antwort, str) or not antwort.strip():
        v.append("antwort_leer"); antwort = ""
    else:
        antwort = antwort.strip()
    roh_q, quellen = obj.get("quellen"), []
    if isinstance(roh_q, str):
        v.append("quellen_string_statt_liste"); roh_q = re.split(r"[,+;\s]+", roh_q)
    if isinstance(roh_q, list):
        for q in roh_q:
            m = re.search(r"0?([1-6])\b", str(q))
            if m:
                quellen.append(f"0{m.group(1)}")
            elif str(q).strip():
                v.append(f"quelle_unbekannt:{str(q)[:12]!r}")
        quellen = sorted(set(quellen))
    else:
        v.append("quellen_fehlen")
    wid = obj.get("widerspruch")
    if isinstance(wid, str):
        v.append("widerspruch_string_statt_bool")
        wid = wid.strip().lower() in ("true", "ja", "yes", "1")
    if not isinstance(wid, bool):
        v.append("widerspruch_fehlt"); wid = False
    if status is None:
        return None, v
    return dict(status=status, antwort=antwort, quellen=quellen, widerspruch=wid), v

def frage_modell(qid, frage, variante):
    treffer = suche(frage)
    messages = baue_prompt(frage, treffer, variante)
    key = cache_key(variante, qid, messages)
    grund = dict(id=qid, variante=variante,
                 top_doks="+".join(t["dok"] for t in treffer),
                 top_abschnitte=" | ".join(f'{t["dok"]}·{t["abschnitt"]}' for t in treffer),
                 beleg_im_kontext=beleg_im_kontext(qid, "\n".join(t["text"] for t in treffer)),
                 bester_score=round(treffer[0]["score"], 3))
    if key in CACHE:
        return {**grund, **CACHE[key], "aus_cache": True}

    letzter, unterwegs = "", []
    for versuch in range(1, CFG["MAX_VERSUCHE"] + 1):
        try:
            r = roher_aufruf(messages, extra=EXTRA or None, response_format=RESPONSE_FORMAT)
            txt = r.choices[0].message.content or ""
            felder, verstoesse = validiere(json_aus_text(txt))
            unterwegs += [f"v{versuch}:{x}" for x in verstoesse]
            if felder is None:
                letzter = "validierung:" + ";".join(verstoesse)
                time.sleep(2 * versuch); continue
            erg = dict(ok=True, **felder, schema_verstoss=";".join(unterwegs),
                       versuche=versuch, out_tokens=r.usage.completion_tokens,
                       in_tokens=r.usage.prompt_tokens, fehler="")
            CACHE[key] = erg
            CACHE_PFAD.write_text(json.dumps(CACHE, ensure_ascii=False))
            return {**grund, **erg, "aus_cache": False}
        except Exception as e:
            letzter = f"{type(e).__name__}: {str(e)[:110]}"
            time.sleep((10 if ("rate" in letzter.lower() or "429" in letzter) else 2) * versuch)
    return {**grund, "ok": False, "status": "", "antwort": "", "quellen": [],
            "widerspruch": False, "schema_verstoss": ";".join(unterwegs),
            "versuche": CFG["MAX_VERSUCHE"], "out_tokens": 0, "in_tokens": 0,
            "fehler": letzter, "aus_cache": False}

def lauf(variante, pause=0.7):
    zeilen = []
    for _, r in eval_df.iterrows():
        res = frage_modell(r["id"], r["frage"], variante)
        res.update(typ=r["typ"], frage=r["frage"],
                   erwartete_antwort=r["erwartete_antwort"], quelle_dok=r["quelle_dok"])
        zeilen.append(res)
        print("c" if res.get("aus_cache") else ("." if res["ok"] else "X"), end="", flush=True)
        if not res.get("aus_cache"):
            time.sleep(pause)
    print()
    return pd.DataFrame(zeilen)

# ---------------------------------------------------------------- Bewertung
def bewerte_zeile(r):
    """(korrekt, fehlercode, notiz). None = Automatik enthaelt sich."""
    if not r["ok"]:
        return None, "api_fehler", str(r.get("fehler", ""))[:80]
    a, status, typ = norm(r["antwort"]), r["status"], r["typ"]
    soll = erwartete_doks(r["quelle_dok"])
    hit = bool(soll & set(str(r["top_doks"]).split("+"))) if soll else None
    regel = PRUEF.get(r["id"], {})

    if typ == "luecke":
        if status in ABGELEHNT:
            return True, "ok", ""
        if status in VORBEHALT:
            return False, "halb_abgelehnt", "unsicher statt klarer Ablehnung"
        return False, "halluziniert", "beantwortet, obwohl nicht dokumentiert"

    if regel.get("nur_manuell"):
        return None, "manuell_pruefen", f"status={status}"

    if status in ABGELEHNT:
        return False, ("unnoetige_ablehnung" if hit is not False
                       else "ablehnung_nach_retrieval_miss"), "abgelehnt, obwohl belegbar"

    fehlt = [p for p in regel.get("muss", []) if not re.search(p, a)]
    tabu  = [p for p in regel.get("tabu", []) if re.search(p, a)]

    if regel.get("braucht_widerspruch") and not r["widerspruch"]:
        if not fehlt and not tabu:
            return False, "widerspruch_nicht_markiert", "beide Seiten genannt, Flag fehlt"
        return False, "widerspruch_verschwiegen", f"fehlende Muster: {len(fehlt)}"

    if fehlt or tabu:
        if r.get("beleg_im_kontext") is False:
            return False, "retrieval_miss", "Beleg stand nicht im Kontext"
        return False, "falscher_wert", f"fehlt: {len(fehlt)}, tabu: {len(tabu)}"

    if soll and not (soll & set(r["quellen"])):
        return True, "quelle_falsch", f"genannt {r['quellen']}, erwartet {sorted(soll)}"
    if [p for p in regel.get("bonus", []) if not re.search(p, a)]:
        return True, "unvollstaendig", "richtig, aber Zusatzangabe fehlt"
    if status in VORBEHALT:
        return True, "ok_mit_vorbehalt", "richtig, aber als unsicher markiert"
    return True, "ok", ""

def bewerte(df):
    d = df.copy()
    res = d.apply(bewerte_zeile, axis=1, result_type="expand")
    d["korrekt_auto"], d["fehlercode_auto"], d["notiz_auto"] = res[0], res[1], res[2]
    soll = d["quelle_dok"].map(erwartete_doks)
    ist  = d["top_doks"].map(lambda s: set(str(s).split("+")))
    d["retrieval_any"]  = [bool(s & i) if s else None for s, i in zip(soll, ist)]
    d["quelle_korrekt"] = [bool(s & set(q)) if s else None for s, q in zip(soll, d["quellen"])]
    d["abgelehnt"]      = d["status"].isin(ABGELEHNT)
    d["n_doks_top3"]    = d["top_doks"].map(lambda s: len(set(str(s).split("+"))))
    for sp in ["korrekt_manuell", "korrekt_final", "fehler_offen",
               "fehlercode_axial", "label_strittig", "label_veraltet"]:
        if sp not in d.columns:
            d[sp] = ""
    return d

# ---------------------------------------------------------------- Kennzahlen
def kennzahlen(d, name, spalte="korrekt_final"):
    n, erfolg = len(d), int(d["ok"].sum())
    print("=" * 74); print(f"{name}   |   erfolgreiche Aufrufe: {erfolg}/{n} = {erfolg/n:.0%}")
    print("=" * 74)
    if erfolg < n:
        print(f"ACHTUNG: {n-erfolg} Aufruf(e) fehlgeschlagen, nicht in den Kennzahlen.")
        for _, r in d[~d["ok"]].iterrows():
            print("  ", r["id"], str(r.get("fehler", ""))[:80])
    ok = d[d["ok"]].copy()
    if spalte not in ok.columns:
        spalte = "korrekt_auto"

    print("\nVerteilung status x typ (Verteilung vor Kennzahl):")
    print(pd.crosstab(ok["typ"], ok["status"]).to_string())
    vs = ok["schema_verstoss"].astype(str)
    if vs.str.len().gt(0).any():
        print("\nSchemaverstoesse trotz response_format:")
        print(ok.loc[vs.str.len() > 0, ["id", "schema_verstoss"]].to_string(index=False))

    b = ok[ok["id"].isin(BEANTWORTBAR)]
    l = ok[ok["id"].isin(LUECKEN)]
    w = ok[ok["id"].isin(WIDERSPRUCH)]
    treffer = b[spalte].map(lambda x: x is True).mean() if len(b) else float("nan")
    ablehn  = l["abgelehnt"].mean() if len(l) else float("nan")
    falsch  = b["abgelehnt"].mean() if len(b) else float("nan")
    ret     = ok.loc[ok["retrieval_any"].notna(), "retrieval_any"].mean()
    bel_s   = ok.loc[ok["beleg_im_kontext"].notna(), "beleg_im_kontext"]
    bel     = bel_s.mean() if len(bel_s) else float("nan")
    beant   = ok[(~ok["abgelehnt"]) & ok["quelle_korrekt"].notna()]
    q_ok    = beant["quelle_korrekt"].mean() if len(beant) else float("nan")

    print(f"\n  Trefferquote    (beantwortbar, n={len(b)}) : {treffer:.0%}")
    print(f"  Ablehnungsquote (luecke,       n={len(l)}) : {ablehn:.0%}")
    print( "  --- die beiden Zahlen NIE zusammenfassen ---")
    print(f"  falsche Ablehnungen bei beantwortbaren    : {falsch:.0%}")
    print(f"  Dokument-Recall (weich)                   : {ret:.0%}")
    print(f"  Beleg-Recall (hart)                       : {bel:.0%}")
    print(f"  Quellenangabe korrekt (nur beantwortete, n={len(beant)}) : {q_ok:.0%}")
    if len(w):
        print(f"  Widerspruch erkannt (n={len(w)})           : {w['widerspruch'].mean():.0%}")
    print("\nFehlercodes (Automatik-Vorschlag):")
    print(ok["fehlercode_auto"].value_counts().to_string())
    return dict(lauf=name, n=n, erfolg=erfolg/n, trefferquote=treffer,
                ablehnungsquote=ablehn, falsche_ablehnung=falsch, dokument_recall=ret,
                beleg_recall=bel, quelle_korrekt=q_ok,
                widerspruch_erkannt=(w["widerspruch"].mean() if len(w) else float("nan")))

print("Definitionen geladen. Legende beim Lauf:  . = neu ok   c = aus Cache   X = fehlgeschlagen")

## 9 — Alle drei Läufe

Eine Zelle, drei Läufe, 75 Aufrufe beim ersten Mal — danach kommt alles aus dem Cache und
kostet nichts. Am Ende liegen im Arbeitsordner:

| Datei | wem gehört sie | wird überschrieben? |
|---|---|---|
| `export_<lauf>.csv` | der Maschine | **ja**, bei jedem Lauf |
| `handcodierung_<lauf>.csv` | **dir** | **nie** — wird nur gelesen |
| `kennzahlen_cs2.csv` | der Maschine | ja |

**Ablauf der Handcodierung:** `export_basis.csv` herunterladen, in Excel bearbeiten, als
`handcodierung_basis.csv` zurück in `cs2_arbeit` legen, Zelle erneut ausführen. Deine
Urteile werden über die Spalte `id` wieder angespielt. Ändert sich eine Antwort, markiert
`label_veraltet` die betroffene Zeile — nur die neu ansehen.

In [ ]:
# ============================================================================
# ZELLE 9 — Laeufe basis / streng / sehr_streng, Bewertung, Export
# ============================================================================
SPALTEN_EXPORT = ["id", "typ", "frage", "erwartete_antwort", "quelle_dok",
                  "status", "antwort", "quellen", "widerspruch",
                  "top_doks", "top_abschnitte", "n_doks_top3",
                  "retrieval_any", "beleg_im_kontext", "quelle_korrekt",
                  "korrekt_auto", "fehlercode_auto", "notiz_auto",
                  "korrekt_manuell", "korrekt_final",
                  "fehler_offen", "fehlercode_axial", "label_strittig", "label_veraltet",
                  "ok", "fehler", "schema_verstoss", "versuche", "in_tokens", "out_tokens"]
HAND_SPALTEN = ["korrekt_manuell", "fehler_offen", "fehlercode_axial", "label_strittig"]
JA, NEIN_ = {"true", "wahr", "ja", "1", "x", "richtig"}, {"false", "falsch", "nein", "0"}

def export_excel(d, pfad, trenner=";"):
    e = d.reindex(columns=SPALTEN_EXPORT).copy()
    e["quellen"] = e["quellen"].map(lambda x: "+".join(x) if isinstance(x, (list, tuple)) else str(x))
    e = e.apply(lambda sp: sp.map(
        lambda v: re.sub(r"[\r\n\t]+", " ", v).replace(trenner, ",").strip()
        if isinstance(v, str) else v))
    e.to_csv(pfad, index=False, sep=trenner, encoding="utf-8-sig")

def uebernehme_handcodierung(d, pfad):
    """Liest DEINE Datei und spielt sie ueber 'id' an. Schreibt nie hinein."""
    pfad = Path(pfad)
    if not pfad.exists():
        print(f"  keine {pfad.name} — alle Urteile aus der Automatik")
        return d
    h = pd.read_csv(pfad, sep=";", dtype=str, encoding="utf-8-sig").fillna("")
    h.columns = [c.strip() for c in h.columns]
    if "id" not in h.columns:
        print(f"  {pfad.name} hat keine Spalte 'id' — ignoriert")
        return d
    h["id"] = h["id"].str.strip()
    for s in [x for x in HAND_SPALTEN if x in h.columns]:
        karte = h.set_index("id")[s].to_dict()
        d[s] = d["id"].map(lambda i: str(karte.get(i, "")).strip())
    if "antwort" in h.columns:
        alt = h.set_index("id")["antwort"].to_dict()
        kurz = lambda t: re.sub(r"[^a-z0-9]", "", str(t).lower())[:120]
        d["label_veraltet"] = ["1" if (str(m).strip() and kurz(alt.get(i, "")) != kurz(a)) else ""
                               for i, a, m in zip(d["id"], d["antwort"], d["korrekt_manuell"])]
    n = int((d["korrekt_manuell"].astype(str).str.len() > 0).sum())
    veraltet = list(d.loc[d["label_veraltet"] == "1", "id"])
    print(f"  Handcodierung aus {pfad.name}: {n} Zeilen mit eigenem Urteil")
    if veraltet:
        print(f"  ACHTUNG, Antwort hat sich seither geaendert: {', '.join(veraltet)}")
    return d

def setze_korrekt_final(d):
    def einer(r):
        m = str(r.get("korrekt_manuell", "")).strip().lower()
        return True if m in JA else (False if m in NEIN_ else r["korrekt_auto"])
    d["korrekt_final"] = d.apply(einer, axis=1)
    hand = int(d["korrekt_manuell"].astype(str).str.strip().str.lower().isin(JA | NEIN_).sum())
    print(f"  korrekt_final: {hand} von Hand, {len(d)-hand} aus der Automatik")
    return d

kpis = []
for variante in ["basis", "streng", "sehr_streng"]:
    print(f"\n### Lauf: {variante}")
    roh = lauf(variante)
    roh.to_csv(ARBEIT / f"roh_{variante}.csv", index=False)
    d = bewerte(roh)
    d = uebernehme_handcodierung(d, ARBEIT / f"handcodierung_{variante}.csv")
    d = setze_korrekt_final(d)
    kpis.append(kennzahlen(d, f"Lauf — {variante}"))
    export_excel(d, ARBEIT / f"export_{variante}.csv")
    print(f"-> {ARBEIT / f'export_{variante}.csv'}")

pd.DataFrame(kpis).to_csv(ARBEIT / "kennzahlen_cs2.csv", index=False)
print(f"\n-> {ARBEIT / 'kennzahlen_cs2.csv'}")
print("\nJetzt: export_basis.csv herunterladen, bearbeiten, als handcodierung_basis.csv")
print("zurueck in cs2_arbeit legen und diese Zelle erneut ausfuehren (Cache = 0 Kosten).")

## 10 — Die Fälle, die Handarbeit brauchen

**Diese Zelle liest nur von der Festplatte.** Sie funktioniert nach einem Neustart der
Laufzeit und auch, wenn Zelle 9 in dieser Sitzung nicht gelaufen ist.

Regel für das Open Coding: **nur der erste Fehler pro Fall**, in `fehler_offen`. Erst danach
verdichten zu kurzen, immer gleich geschriebenen Codes in `fehlercode_axial`.

In [ ]:
# ============================================================================
# ZELLE 10 — Faelle zur Handcodierung (liest nur Dateien, keine Variablen)
# ============================================================================
import textwrap
from pathlib import Path
import pandas as pd

LAUF_ANSEHEN = "basis"      # "basis" | "streng" | "sehr_streng"

def finde(name):
    for p in [Path("cs2_arbeit") / name, Path("/content/cs2_arbeit") / name, Path(name)]:
        if p.is_file():
            return p
    return None

datei = finde(f"export_{LAUF_ANSEHEN}.csv")
if datei is None:
    print(f"export_{LAUF_ANSEHEN}.csv nicht gefunden.")
    print("Zelle 9 laufen lassen, oder die Datei nach cs2_arbeit/ hochladen.")
else:
    d = pd.read_csv(datei, sep=";", dtype=str, encoding="utf-8-sig").fillna("")
    tf = lambda v: True if str(v).strip().lower() == "true" else (
        False if str(v).strip().lower() == "false" else None)
    for sp in ["korrekt_auto", "korrekt_final", "retrieval_any", "beleg_im_kontext"]:
        if sp in d.columns:
            d[sp] = d[sp].map(tf)
    if "korrekt_final" not in d.columns:
        d["korrekt_final"] = d.get("korrekt_auto")

    zu_pruefen = d[(d["korrekt_final"] != True) | (d["fehlercode_auto"] != "ok")]
    print(f"Quelle: {datei}")
    print(f"{len(zu_pruefen)} von {len(d)} Faellen brauchen einen Blick.\n")
    for _, r in zu_pruefen.iterrows():
        print("-" * 74)
        print(f"{r['id']} [{r['typ']}] {r['frage']}")
        print(f"  erwartet : {r['erwartete_antwort']}")
        print(f"  status   : {r['status']}   quellen: {r['quellen']}   widerspruch: {r['widerspruch']}")
        print(f"  antwort  : {textwrap.shorten(str(r['antwort']), 260)}")
        print(f"  kontext  : {r.get('top_abschnitte', '—')}")
        print(f"  retrieval: dokument={r.get('retrieval_any')}  beleg={r.get('beleg_im_kontext')}"
              f"   (erwartet {r['quelle_dok']})")
        print(f"  auto     : korrekt={r['korrekt_auto']}  code={r['fehlercode_auto']}  {r['notiz_auto']}")

## 11 — Vergleich, Diagramm, Kosten

**Liest ebenfalls nur von der Festplatte.** Erzeugt `cs2_zielkonflikt.png` — das Bild für
die Website.

In [ ]:
# ============================================================================
# ZELLE 11 — Vergleich, Diagramm, Kosten (liest nur Dateien)
# ============================================================================
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _finde(name):
    for p in [Path("cs2_arbeit") / name, Path("/content/cs2_arbeit") / name, Path(name)]:
        if p.is_file():
            return p
    return None

kz = _finde("kennzahlen_cs2.csv")
if kz is None:
    print("kennzahlen_cs2.csv nicht gefunden — bitte Zelle 9 laufen lassen.")
else:
    ORDNER = kz.parent
    v = pd.read_csv(kz)
    v["kurz"] = v["lauf"].str.replace("Lauf — ", "", regex=False)
    pz = lambda x: f"{x:.0%}" if pd.notna(x) else "—"
    spalten = [c for c in ["trefferquote", "ablehnungsquote", "falsche_ablehnung",
                           "dokument_recall", "beleg_recall", "quelle_korrekt",
                           "widerspruch_erkannt", "erfolg"] if c in v.columns]
    print("=" * 74)
    print("ZIELKONFLIKT — Trefferquote gegen Ablehnungsquote")
    print("=" * 74)
    print(v.set_index("kurz")[spalten].apply(lambda s: s.map(pz)).to_string())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8))
    ax1.plot(v["ablehnungsquote"] * 100, v["trefferquote"] * 100, "-o",
             color="#3b6ea5", linewidth=1.6, markersize=9, zorder=3)
    for _, r in v.iterrows():
        ax1.annotate(r["kurz"], (r["ablehnungsquote"] * 100, r["trefferquote"] * 100),
                     textcoords="offset points", xytext=(0, 12), ha="center", fontsize=10)
    ax1.set_xlabel("Ablehnungsquote bei echten Lücken (%)")
    ax1.set_ylabel("Trefferquote bei beantwortbaren Fragen (%)")
    ax1.set_title("Je strenger die Ablehnungsregel,\ndesto teurer die Vorsicht",
                  fontsize=11, loc="left")
    ax1.set_xlim(-12, 112); ax1.set_ylim(-6, 114)
    ax1.grid(alpha=0.25); ax1.set_axisbelow(True)

    codes, tokens = {}, []
    for name in v["kurz"]:
        p = _finde(f"export_{name}.csv")
        if p is None:
            continue
        e = pd.read_csv(p, sep=";", dtype=str, encoding="utf-8-sig").fillna("")
        codes[name] = e.loc[e["ok"].str.lower() == "true", "fehlercode_auto"].value_counts()
        tokens.append(e.loc[e["ok"].str.lower() == "true",
                            ["in_tokens", "out_tokens"]].astype(float).sum())

    if codes:
        c = pd.DataFrame(codes).fillna(0)
        c = c.drop(index=[i for i in ["ok", "ok_mit_vorbehalt"] if i in c.index], errors="ignore")
        c = c.loc[c.sum(axis=1).sort_values().index]
        palette = ["#3b6ea5", "#c98b3a", "#8a5a83", "#4f8a6d"]
        c.plot(kind="barh", ax=ax2, width=0.75,
               color=[palette[i % len(palette)] for i in range(c.shape[1])])
        ax2.set_title("Fehlerklassen je Lauf (Automatik-Vorschlag)", fontsize=11, loc="left")
        ax2.set_xlabel("Fälle"); ax2.set_ylabel("")
        ax2.grid(axis="x", alpha=0.25); ax2.set_axisbelow(True)
        ax2.legend(fontsize=9)

    fig.suptitle("VoltRide Support-Assistent (fiktive Daten) — Grounding gegen Ablehnung",
                 fontsize=12, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(ORDNER / "cs2_zielkonflikt.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"\n-> {ORDNER / 'cs2_zielkonflikt.png'}")

    if tokens:
        t = pd.concat(tokens, axis=1).sum(axis=1)
        PREIS_IN, PREIS_OUT = 0.10, 0.50   # USD je 1 Mio Token, vor Veroeffentlichung pruefen
        k = t["in_tokens"] / 1e6 * PREIS_IN + t["out_tokens"] / 1e6 * PREIS_OUT
        print(f"\nKosten ueber alle Laeufe: {k:.4f} USD "
              f"({int(t['in_tokens']):,} in / {int(t['out_tokens']):,} out)")
        print(f"  Hochrechnung 200 Fragen/Tag = 6.000/Monat: "
              f"{k/len(v)/25*6000:.2f} USD/Monat")
        print( "  Embeddings: 0 USD (lokal). Der Aufwand liegt in der Pflege der Dokumente,")
        print( "  nicht im Modell.")

## Protokolltabelle

| Lauf | Eingriff | Erwartung vorher | Beleg-Recall | Trefferquote | Ablehnungsquote | Überraschung |
|---|---|---|---|---|---|---|
| 1 | `basis` | | | | | |
| 2 | `streng` | | | | | |
| 3 | `sehr_streng` | | | | | |
| 4 | *(dein Retrieval-Eingriff)* | | | | | |

**Ein Eingriff pro Lauf.** Erwartung vorher aufschreiben, danach vergleichen — nicht
danach umdeuten. Codes gelten nur für den Lauf, aus dem sie stammen.

**Nach jedem Lauf:** den Ordner `cs2_arbeit` herunterladen. Colab löscht ihn beim Neustart
der Laufzeit, und mit ihm den Cache und deine Handcodierung.